# 02. Advanced Data Cleaning & Feature Engineering

## Objectives:
1. **Anomaly Correction**: Address negative values in usage and billing columns.
2. **Outlier Detection**: Identify and handle extreme values in `estimated_salary` and `data_used`.
3. **Feature Engineering**: Create meaningful segments like `Age Group` and `Usage Tiers`.
4. **Standardization**: Ensure categorical consistency across the dataset.

In [1]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

# Aesthetic Setup
sns.set_palette("husl")
sns.set_style("whitegrid")

# Load the raw data
raw_path = "../data/raw/telecom_churn_raw_dataset.csv"
df = pd.read_csv(raw_path)
print(f"Initial Dataset Load: {df.shape[0]:,} rows and {df.shape[1]} columns.")

Initial Dataset Load: 243,553 rows and 14 columns.


## 1. Correcting Negative Anomalies
We observed negative values in numerical columns. These are likely entry errors. We will convert them to absolute values to preserve the magnitude of the data point.

In [2]:
numerical_cols = ['calls_made', 'sms_sent', 'data_used', 'estimated_salary', 'age']

for col in numerical_cols:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"[FIX] Found {neg_count} negative values in '{col}'. Applying absolute transformation.")
        df[col] = df[col].abs()

print("\nAll numerical columns are now non-negative.")

[FIX] Found 6713 negative values in 'calls_made'. Applying absolute transformation.
[FIX] Found 7375 negative values in 'sms_sent'. Applying absolute transformation.
[FIX] Found 6050 negative values in 'data_used'. Applying absolute transformation.

All numerical columns are now non-negative.


## 2. Feature Engineering
We will create derived features to help our analysis:
- **Age Group**: Categorizing customers into Life Stages.
- **Usage Density**: A metric combining calls, SMS, and data.

In [3]:
# 1. Age Binning
bins = [18, 30, 45, 60, 100]
labels = ['Young Adult', 'Middle Aged', 'Senior', 'Elderly']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

# 2. Registration Date processing
df['date_of_registration'] = pd.to_datetime(df['date_of_registration'])
df['registration_year'] = df['date_of_registration'].dt.year
df['registration_month'] = df['date_of_registration'].dt.month_name()

# 3. Total Usage Score (Standardized)
df['usage_score'] = (df['calls_made'] * 0.4) + (df['sms_sent'] * 0.1) + (df['data_used'] * 0.5)

print("Feature engineering complete.")

Feature engineering complete.


## 3. Saving the Cleaned Dataset
We store the results in `data/processed/` for downstream modeling and EDA.

In [4]:
processed_dir = "../data/processed"
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

save_path = os.path.join(processed_dir, "telecom_churn_cleaned.csv")
df.to_csv(save_path, index=False)

print(f"SUCCESS: Cleaned dataset saved to {save_path}")

SUCCESS: Cleaned dataset saved to ../data/processed/telecom_churn_cleaned.csv
